In [1]:
from pathlib import Path
import json
import random
import shutil
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import datasets, models, transforms
from PIL import Image

In [2]:
REALWASTE_DIR = Path(r"E:\SPCK-CSI18\data\RealWaste_split")

for split in ["train", "val", "test"]:
    print(f"\n{split.upper()}")

    split_dir = REALWASTE_DIR / split

    for class_dir in sorted(split_dir.iterdir()):
        if class_dir.is_dir():
            count = sum(
                1 for file in class_dir.rglob("*")
                if file.is_file()
                and file.suffix.lower() in {
                    ".jpg", ".jpeg", ".png", ".webp", ".bmp"
                }
            )
            print(f"{class_dir.name}: {count}")


TRAIN
Cardboard: 322
Food Organics: 287
Glass: 294
Metal: 553
Miscellaneous Trash: 346
Paper: 350
Plastic: 644
Textile Trash: 222
Vegetation: 305

VAL
Cardboard: 46
Food Organics: 41
Glass: 42
Metal: 79
Miscellaneous Trash: 49
Paper: 50
Plastic: 92
Textile Trash: 31
Vegetation: 43

TEST
Cardboard: 93
Food Organics: 83
Glass: 84
Metal: 158
Miscellaneous Trash: 100
Paper: 100
Plastic: 185
Textile Trash: 65
Vegetation: 88


In [3]:
PROJECT_DIR = Path(r"E:\SPCK-CSI18")

TEXTILE_ROOT = Path(r"data\textile")
TEXTILE_TRAIN_DIR = Path(r"data\textile\train")
TEXTILE_VAL_DIR = Path(r"data\textile\valid")
TEXTILE_TEST_DIR = Path(r"data\textile\test")

print(Path.cwd())

e:\SPCK-CSI18


In [8]:
TEXTILE_SPLITS2 = {
    "train": Path(r"data\textile\train"),
    "val": Path(r"data\textile\valid"),
    "test": Path(r"data\textile\test"),
}

CLASS_NAMES2 = [
    "battery", "biological", "cardboard", "clothes", "glass",
    "metal", "paper", "plastic", "shoes", "trash"
]

TEXTILE_CLASS_MAP2 = {
    "Clothes": "clothes",
    "Textile Waste": "clothes",
}

textile_samples2 = {
    "train": [],
    "val": [],
    "test": [],
}

for split_name, split_dir in TEXTILE_SPLITS2.items():
    json_path = split_dir / "_annotations.coco.json"

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    categories2 = {
        category["id"]: category["name"]
        for category in data["categories"]
    }

    images2 = {
        image["id"]: image
        for image in data["images"]
    }

    annotations_by_image2 = {}

    for ann in data["annotations"]:
        annotations_by_image2.setdefault(
            ann["image_id"], []
        ).append(ann)

    for image_id, anns in annotations_by_image2.items():
        image_info = images2.get(image_id)

        if image_info is None:
            continue

        image_path = split_dir / image_info["file_name"]

        if not image_path.is_file():
            image_path = next(
                (
                    p for p in split_dir.rglob(
                        Path(image_info["file_name"]).name
                    )
                    if p.is_file()
                ),
                None
            )

        if image_path is None or not image_path.is_file():
            continue

        for ann in anns:
            category_name = categories2.get(ann["category_id"])
            target_class = TEXTILE_CLASS_MAP2.get(category_name)

            if target_class != "clothes":
                continue

            x, y, w, h = ann["bbox"]

            if w <= 0 or h <= 0:
                continue

            textile_samples2[split_name].append({
                "image_path": image_path,
                "bbox": (x, y, w, h),
                "label": CLASS_NAMES2.index("clothes"),
                "image_id": image_id,
                "source": "textile",
            })

    print(
        f"{split_name.upper()}: "
        f"{len(textile_samples2[split_name])} clothing samples"
    )

print(
    "\nTotal Textile Waste samples:",
    sum(len(v) for v in textile_samples2.values())
)

TRAIN: 392 clothing samples
VAL: 113 clothing samples
TEST: 57 clothing samples

Total Textile Waste samples: 562


In [9]:
CLASS_NAMES2 = [
    "battery", "biological", "cardboard", "clothes", "glass",
    "metal", "paper", "plastic", "shoes", "trash"
]

REALWASTE_MAP2 = {
    "Cardboard": "cardboard",
    "Food Organics": "biological",
    "Glass": "glass",
    "Metal": "metal",
    "Miscellaneous Trash": "trash",
    "Paper": "paper",
    "Plastic": "plastic",
    "Textile Trash": "clothes",
    "Vegetation": "biological",
}

REALWASTE_SPLITS2 = {
    "train": Path(r"data\RealWaste_split\train"),
    "val": Path(r"data\RealWaste_split\val"),
    "test": Path(r"data\RealWaste_split\test"),
}

IMAGE_EXTENSIONS2 = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

realwaste_samples2 = {
    "train": [],
    "val": [],
    "test": [],
}

for split_name, split_dir in REALWASTE_SPLITS2.items():
    if not split_dir.is_dir():
        raise FileNotFoundError(f"Không tìm thấy: {split_dir}")

    for class_dir in split_dir.iterdir():
        if not class_dir.is_dir():
            continue

        target_class = REALWASTE_MAP2.get(class_dir.name)

        if target_class is None:
            print("Bỏ qua class chưa ánh xạ:", class_dir.name)
            continue

        for image_path in class_dir.rglob("*"):
            if not image_path.is_file():
                continue

            if image_path.suffix.lower() not in IMAGE_EXTENSIONS2:
                continue

            realwaste_samples2[split_name].append({
                "image_path": image_path,
                "bbox": None,
                "label": CLASS_NAMES2.index(target_class),
                "image_id": str(image_path.resolve()),
                "source": "realwaste",
            })

    print(
        f"{split_name.upper()}: "
        f"{len(realwaste_samples2[split_name])} ảnh"
    )

TRAIN: 3323 ảnh
VAL: 473 ảnh
TEST: 956 ảnh


In [10]:
PROJECT_DIR2 = Path.cwd()

V2_DIR2 = Path(r"data\dataset")
REALWASTE_DIR2 = Path(r"data\RealWaste_split")
TEXTILE_DIR2 = Path(r"data\textile")

IMAGE_EXTENSIONS2 = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

print("Project:", PROJECT_DIR2)

dataset_dirs2 = {
    "V2 train": V2_DIR2 / "train",
    "V2 val": V2_DIR2 / "val",
    "V2 test": V2_DIR2 / "test",
    "RealWaste train": REALWASTE_DIR2 / "train",
    "RealWaste val": REALWASTE_DIR2 / "val",
    "RealWaste test": REALWASTE_DIR2 / "test",
    "Textile train": TEXTILE_DIR2 / "train",
    "Textile val": TEXTILE_DIR2 / "valid",
    "Textile test": TEXTILE_DIR2 / "test",
}

for name, folder in dataset_dirs2.items():
    if not folder.is_dir():
        print(f"{name}: NOT FOUND - {folder}")
        continue

    count = sum(
        1 for file in folder.rglob("*")
        if file.is_file()
        and file.suffix.lower() in IMAGE_EXTENSIONS2
    )

    print(f"{name}: {count} images")

taco_train_files2 = [
    file for file in PROJECT_DIR2.rglob("train.json")
    if file.parent.name == "splits"
    and (file.parent / "val.json").is_file()
    and (file.parent / "test.json").is_file()
]

print("\nTACO split files found:", len(taco_train_files2))

for file in taco_train_files2:
    print("TACO splits:", file.parent)

Project: e:\SPCK-CSI18
V2 train: 9801 images
V2 val: 1227 images
V2 test: 1230 images
RealWaste train: 3323 images
RealWaste val: 473 images
RealWaste test: 956 images
Textile train: 393 images
Textile val: 113 images
Textile test: 56 images

TACO split files found: 1
TACO splits: e:\SPCK-CSI18\data\TACO data\splits


In [11]:
CLASS_NAMES2 = [
    "battery", "biological", "cardboard", "clothes", "glass",
    "metal", "paper", "plastic", "shoes", "trash"
]

REALWASTE_MAP2 = {
    "Cardboard": "cardboard",
    "Food Organics": "biological",
    "Glass": "glass",
    "Metal": "metal",
    "Miscellaneous Trash": "trash",
    "Paper": "paper",
    "Plastic": "plastic",
    "Textile Trash": "clothes",
    "Vegetation": "biological",
}

TACO_MAP2 = {
    "Battery": "battery",
    "Food waste": "biological",
    "Other carton": "cardboard",
    "Egg carton": "cardboard",
    "Corrugated carton": "cardboard",
    "Pizza box": "cardboard",
    "Drink carton": "cardboard",
    "Meal carton": "cardboard",
    "Toilet tube": "cardboard",
    "Glass bottle": "glass",
    "Broken glass": "glass",
    "Glass cup": "glass",
    "Glass jar": "glass",
    "Aluminium foil": "metal",
    "Metal bottle cap": "metal",
    "Food Can": "metal",
    "Aerosol": "metal",
    "Drink can": "metal",
    "Metal lid": "metal",
    "Pop tab": "metal",
    "Scrap metal": "metal",
    "Magazine paper": "paper",
    "Tissues": "paper",
    "Wrapping paper": "paper",
    "Normal paper": "paper",
    "Paper bag": "paper",
    "Paper cup": "paper",
    "Paper straw": "paper",
    "Other plastic bottle": "plastic",
    "Clear plastic bottle": "plastic",
    "Plastic bottle cap": "plastic",
    "Disposable plastic cup": "plastic",
    "Foam cup": "plastic",
    "Other plastic cup": "plastic",
    "Plastic lid": "plastic",
    "Other plastic": "plastic",
    "Plastic film": "plastic",
    "Six pack rings": "plastic",
    "Garbage bag": "plastic",
    "Other plastic wrapper": "plastic",
    "Single-use carrier bag": "plastic",
    "Polypropylene bag": "plastic",
    "Crisp packet": "plastic",
    "Spread tub": "plastic",
    "Tupperware": "plastic",
    "Disposable food container": "plastic",
    "Foam food container": "plastic",
    "Other plastic container": "plastic",
    "Plastic glooves": "plastic",
    "Plastic utensils": "plastic",
    "Plastic straw": "plastic",
    "Styrofoam piece": "plastic",
    "Squeezable tube": "plastic",
    "Shoe": "shoes",
    "Cigarette": "trash",
}

TEXTILE_MAP2 = {
    "Clothes": "clothes",
    "Textile Waste": "clothes",
}

V2_SPLITS2 = {
    "train": Path(r"data\dataset\train"),
    "val": Path(r"data\dataset\val"),
    "test": Path(r"data\dataset\test"),
}

REALWASTE_SPLITS2 = {
    "train": Path(r"data\RealWaste_split\train"),
    "val": Path(r"data\RealWaste_split\val"),
    "test": Path(r"data\RealWaste_split\test"),
}

TEXTILE_SPLITS2 = {
    "train": Path(r"data\textile\train"),
    "val": Path(r"data\textile\valid"),
    "test": Path(r"data\textile\test"),
}

TACO_SPLITS2 = {
    "train": Path(r"data\TACO data\splits\train.json"),
    "val": Path(r"data\TACO data\splits\val.json"),
    "test": Path(r"data\TACO data\splits\test.json"),
}

IMAGE_EXTENSIONS2 = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

print("V2 class folders:")
v2_classes2 = sorted(
    folder.name
    for folder in V2_SPLITS2["train"].iterdir()
    if folder.is_dir()
)
print(v2_classes2)

if set(v2_classes2) != set(CLASS_NAMES2):
    print("WARNING: V2 folders do not match the expected 10 classes.")

print("\nRealWaste and V2 label mapping:")
for original, target in REALWASTE_MAP2.items():
    print(f"{original} -> {target}")

print("\nTextile Waste mapping:")
for original, target in TEXTILE_MAP2.items():
    print(f"{original} -> {target}")

print("\nTACO mapped categories:", len(TACO_MAP2))

for split_name, json_path in TACO_SPLITS2.items():
    if not json_path.is_file():
        raise FileNotFoundError(f"TACO split not found: {json_path}")

    with open(json_path, "r", encoding="utf-8") as f:
        taco_split2 = json.load(f)

    category_names2 = {
        category["id"]: category["name"]
        for category in taco_split2["categories"]
    }

    counts2 = Counter()

    for ann in taco_split2["annotations"]:
        category_name = category_names2.get(ann["category_id"])
        target = TACO_MAP2.get(category_name)

        if target is not None:
            counts2[target] += 1

    print(f"\nTACO {split_name.upper()} annotation counts:")
    for class_name in CLASS_NAMES2:
        print(f"{class_name}: {counts2[class_name]}")

print("\nLabel mapping defined. No files were changed.")

V2 class folders:
['battery', 'biological', 'cardboard', 'clothes', 'glass', 'metal', 'paper', 'plastic', 'shoes', 'trash']

RealWaste and V2 label mapping:
Cardboard -> cardboard
Food Organics -> biological
Glass -> glass
Metal -> metal
Miscellaneous Trash -> trash
Paper -> paper
Plastic -> plastic
Textile Trash -> clothes
Vegetation -> biological

Textile Waste mapping:
Clothes -> clothes
Textile Waste -> clothes

TACO mapped categories: 55

TACO TRAIN annotation counts:
battery: 1
biological: 5
cardboard: 173
clothes: 0
glass: 79
metal: 330
paper: 143
plastic: 1382
shoes: 6
trash: 320

TACO VAL annotation counts:
battery: 1
biological: 0
cardboard: 13
clothes: 0
glass: 39
metal: 53
paper: 30
plastic: 161
shoes: 1
trash: 50

TACO TEST annotation counts:
battery: 0
biological: 3
cardboard: 35
clothes: 0
glass: 20
metal: 74
paper: 42
plastic: 391
shoes: 0
trash: 171

Label mapping defined. No files were changed.


In [12]:
samples_by_source2 = {
    source: {split: [] for split in ["train", "val", "test"]}
    for source in ["v2", "taco", "realwaste", "textile"]
}

missing_samples2 = Counter()

for split_name, split_dir in V2_SPLITS2.items():
    for class_name in CLASS_NAMES2:
        class_dir = split_dir / class_name

        for image_path in class_dir.rglob("*"):
            if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS2:
                samples_by_source2["v2"][split_name].append({
                    "image_path": image_path,
                    "bbox": None,
                    "label": CLASS_NAMES2.index(class_name),
                    "source": "v2",
                })

for split_name, split_dir in REALWASTE_SPLITS2.items():
    for class_dir in split_dir.iterdir():
        if not class_dir.is_dir():
            continue

        target_class = REALWASTE_MAP2.get(class_dir.name)
        if target_class is None:
            continue

        for image_path in class_dir.rglob("*"):
            if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS2:
                samples_by_source2["realwaste"][split_name].append({
                    "image_path": image_path,
                    "bbox": None,
                    "label": CLASS_NAMES2.index(target_class),
                    "source": "realwaste",
                })

for split_name, json_path in TACO_SPLITS2.items():
    with open(json_path, "r", encoding="utf-8") as f:
        taco_split2 = json.load(f)

    categories2 = {
        c["id"]: c["name"] for c in taco_split2["categories"]
    }
    images2 = {
        im["id"]: im for im in taco_split2["images"]
    }

    for ann in taco_split2["annotations"]:
        category_name = categories2.get(ann["category_id"])
        target_class = TACO_MAP2.get(category_name)

        if target_class is None:
            continue

        image_info = images2.get(ann["image_id"])
        if image_info is None:
            continue

        image_path = TACO_SPLITS2["train"].parent.parent / image_info["file_name"]

        if not image_path.is_file():
            missing_samples2["taco"] += 1
            continue

        x, y, w, h = ann["bbox"]
        if w <= 0 or h <= 0:
            continue

        samples_by_source2["taco"][split_name].append({
            "image_path": image_path,
            "bbox": (x, y, w, h),
            "label": CLASS_NAMES2.index(target_class),
            "source": "taco",
            "image_id": ann["image_id"],
        })

for split_name, split_dir in TEXTILE_SPLITS2.items():
    json_path = split_dir / "_annotations.coco.json"

    with open(json_path, "r", encoding="utf-8") as f:
        textile_data2 = json.load(f)

    categories2 = {
        c["id"]: c["name"] for c in textile_data2["categories"]
    }
    images2 = {
        im["id"]: im for im in textile_data2["images"]
    }

    files_by_name2 = {
        p.name: p
        for p in split_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS2
    }

    for ann in textile_data2["annotations"]:
        category_name = categories2.get(ann["category_id"])
        target_class = TEXTILE_MAP2.get(category_name)

        if target_class is None:
            continue

        image_info = images2.get(ann["image_id"])
        if image_info is None:
            continue

        image_path = split_dir / image_info["file_name"]

        if not image_path.is_file():
            image_path = files_by_name2.get(
                Path(image_info["file_name"]).name
            )

        if image_path is None or not image_path.is_file():
            missing_samples2["textile"] += 1
            continue

        x, y, w, h = ann["bbox"]
        if w <= 0 or h <= 0:
            continue

        samples_by_source2["textile"][split_name].append({
            "image_path": image_path,
            "bbox": (x, y, w, h),
            "label": CLASS_NAMES2.index(target_class),
            "source": "textile",
            "image_id": ann["image_id"],
        })

for split_name in ["train", "val", "test"]:
    print(f"\n{'=' * 35}\n{split_name.upper()}")

    for source in ["v2", "taco", "realwaste", "textile"]:
        samples = samples_by_source2[source][split_name]
        counts = Counter(
            CLASS_NAMES2[sample["label"]]
            for sample in samples
        )

        print(f"\n{source}: {len(samples)} samples")
        print({
            class_name: counts[class_name]
            for class_name in CLASS_NAMES2
            if counts[class_name] > 0
        })

print("\nMissing samples:", dict(missing_samples2))
print("\nData preparation lists created. Training has not started.")


TRAIN

v2: 9801 samples
{'battery': 604, 'biological': 559, 'cardboard': 1128, 'clothes': 1513, 'glass': 1388, 'metal': 744, 'paper': 1068, 'plastic': 1277, 'shoes': 1159, 'trash': 361}

taco: 2439 samples
{'battery': 1, 'biological': 5, 'cardboard': 173, 'glass': 79, 'metal': 330, 'paper': 143, 'plastic': 1382, 'shoes': 6, 'trash': 320}

realwaste: 3323 samples
{'biological': 592, 'cardboard': 322, 'clothes': 222, 'glass': 294, 'metal': 553, 'paper': 350, 'plastic': 644, 'trash': 346}

textile: 392 samples
{'clothes': 392}

VAL

v2: 1227 samples
{'battery': 76, 'biological': 70, 'cardboard': 141, 'clothes': 189, 'glass': 174, 'metal': 93, 'paper': 134, 'plastic': 160, 'shoes': 145, 'trash': 45}

taco: 348 samples
{'battery': 1, 'cardboard': 13, 'glass': 39, 'metal': 53, 'paper': 30, 'plastic': 161, 'shoes': 1, 'trash': 50}

realwaste: 473 samples
{'biological': 84, 'cardboard': 46, 'clothes': 31, 'glass': 42, 'metal': 79, 'paper': 50, 'plastic': 92, 'trash': 49}

textile: 113 samples

In [13]:
split_samples2 = {}

for split_name in ["train", "val", "test"]:
    split_samples2[split_name] = []

    for source in ["v2", "taco", "realwaste", "textile"]:
        split_samples2[split_name].extend(
            samples_by_source2[source][split_name]
        )

train_transform2 = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

eval_transform2 = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])


class WasteDataset2(Dataset):
    def __init__(self, samples, transform):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        sample = self.samples[index]

        with Image.open(sample["image_path"]) as source:
            if "A" in source.getbands() or "transparency" in source.info:
                rgba = source.convert("RGBA")
                background = Image.new(
                    "RGBA", rgba.size, (245, 245, 245, 255)
                )
                image = Image.alpha_composite(
                    background, rgba
                ).convert("RGB")
            else:
                image = source.convert("RGB")

            bbox = sample.get("bbox")

            if bbox is not None:
                x, y, w, h = bbox
                image_width, image_height = image.size

                pad_x = w * 0.20
                pad_y = h * 0.20

                left = max(0, int(x - pad_x))
                top = max(0, int(y - pad_y))
                right = min(image_width, int(x + w + pad_x))
                bottom = min(image_height, int(y + h + pad_y))

                if right <= left or bottom <= top:
                    raise ValueError(
                        f"Invalid bounding box: {sample['image_path']}"
                    )

                image = image.crop((left, top, right, bottom))

        image = self.transform(image)

        return image, sample["label"], sample["source"]


train_dataset2 = WasteDataset2(
    split_samples2["train"], train_transform2
)
val_dataset2 = WasteDataset2(
    split_samples2["val"], eval_transform2
)
test_dataset2 = WasteDataset2(
    split_samples2["test"], eval_transform2
)

BATCH_SIZE2 = 16

train_loader2 = DataLoader(
    train_dataset2,
    batch_size=BATCH_SIZE2,
    shuffle=True,
    num_workers=0
)

val_loader2 = DataLoader(
    val_dataset2,
    batch_size=BATCH_SIZE2,
    shuffle=False,
    num_workers=0
)

test_loader2 = DataLoader(
    test_dataset2,
    batch_size=BATCH_SIZE2,
    shuffle=False,
    num_workers=0
)

print("Train samples:", len(train_dataset2))
print("Validation samples:", len(val_dataset2))
print("Test samples:", len(test_dataset2))

images2, labels2, sources2 = next(iter(train_loader2))

print("\nBatch check:")
print("Images shape:", images2.shape)
print("Labels shape:", labels2.shape)
print("Labels in batch:", labels2.tolist())
print("Sources in batch:", list(sources2))

Train samples: 15955
Validation samples: 2161
Test samples: 2979

Batch check:
Images shape: torch.Size([16, 3, 256, 256])
Labels shape: torch.Size([16])
Labels in batch: [6, 2, 7, 1, 7, 6, 2, 7, 9, 5, 5, 4, 6, 4, 9, 9]
Sources in batch: ['realwaste', 'v2', 'taco', 'realwaste', 'taco', 'realwaste', 'v2', 'v2', 'v2', 'taco', 'v2', 'v2', 'realwaste', 'v2', 'realwaste', 'v2']


In [15]:
for split_name2 in ["train", "val", "test"]:
    valid_samples2 = []
    invalid_boxes2 = 0
    unreadable_images2 = 0

    for sample2 in split_samples2[split_name2]:
        try:
            with Image.open(sample2["image_path"]) as image2:
                width2, height2 = image2.size
        except Exception:
            unreadable_images2 += 1
            continue

        bbox2 = sample2.get("bbox")

        if bbox2 is not None:
            x2, y2, w2, h2 = map(float, bbox2)

            if (
                not np.isfinite([x2, y2, w2, h2]).all()
                or w2 <= 0
                or h2 <= 0
            ):
                invalid_boxes2 += 1
                continue

            left2 = max(0.0, x2)
            top2 = max(0.0, y2)
            right2 = min(float(width2), x2 + w2)
            bottom2 = min(float(height2), y2 + h2)

            if right2 <= left2 or bottom2 <= top2:
                invalid_boxes2 += 1
                continue

            sample2 = sample2.copy()
            sample2["bbox"] = (
                left2,
                top2,
                right2 - left2,
                bottom2 - top2,
            )

        valid_samples2.append(sample2)

    split_samples2[split_name2] = valid_samples2

    print(
        f"{split_name2.upper()}: "
        f"{len(valid_samples2)} valid samples | "
        f"{invalid_boxes2} invalid boxes removed | "
        f"{unreadable_images2} unreadable images"
    )

TRAIN: 15912 valid samples | 43 invalid boxes removed | 0 unreadable images
VAL: 2156 valid samples | 5 invalid boxes removed | 0 unreadable images
TEST: 2946 valid samples | 33 invalid boxes removed | 0 unreadable images


In [16]:
train_dataset2 = WasteDataset2(
    split_samples2["train"], train_transform2
)
val_dataset2 = WasteDataset2(
    split_samples2["val"], eval_transform2
)
test_dataset2 = WasteDataset2(
    split_samples2["test"], eval_transform2
)

train_loader2 = DataLoader(
    train_dataset2,
    batch_size=16,
    shuffle=True,
    num_workers=0,
)

val_loader2 = DataLoader(
    val_dataset2,
    batch_size=16,
    shuffle=False,
    num_workers=0,
)

test_loader2 = DataLoader(
    test_dataset2,
    batch_size=16,
    shuffle=False,
    num_workers=0,
)

print("DataLoaders rebuilt successfully.")

DataLoaders rebuilt successfully.


In [17]:
device2 = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

MODEL_BEST2 = Path("garbage_classifier_4datasets_best.pth")
CHECKPOINT2 = Path("garbage_classifier_4datasets_checkpoint.pth")
CLASS_NAMES_PATH2 = Path("garbage_class_names_10.json")

with open(CLASS_NAMES_PATH2, "w", encoding="utf-8") as f:
    json.dump(CLASS_NAMES2, f, ensure_ascii=False, indent=2)

model2 = models.mobilenet_v3_small(
    weights=models.MobileNet_V3_Small_Weights.DEFAULT
)

model2.classifier[-1] = nn.Linear(
    model2.classifier[-1].in_features,
    len(CLASS_NAMES2)
)

model2 = model2.to(device2)

train_counts2 = np.bincount(
    [sample["label"] for sample in split_samples2["train"]],
    minlength=len(CLASS_NAMES2)
)

class_weights2 = 1.0 / np.sqrt(np.maximum(train_counts2, 1))
class_weights2 = class_weights2 / class_weights2.mean()

criterion2 = nn.CrossEntropyLoss(
    weight=torch.tensor(
        class_weights2,
        dtype=torch.float32,
        device=device2
    )
)

optimizer2 = optim.AdamW(
    model2.parameters(),
    lr=0.0001,
    weight_decay=0.01
)

scheduler2 = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer2,
    mode="min",
    factor=0.5,
    patience=2,
    min_lr=1e-6
)

EPOCHS2 = 20
PATIENCE2 = 5
best_macro_acc2 = -1.0
best_val_loss2 = float("inf")
early_stop_counter2 = 0
start_epoch2 = 0

if CHECKPOINT2.exists():
    checkpoint_data2 = torch.load(
        CHECKPOINT2,
        map_location=device2,
        weights_only=True
    )

    model2.load_state_dict(checkpoint_data2["model"])
    optimizer2.load_state_dict(checkpoint_data2["optimizer"])
    scheduler2.load_state_dict(checkpoint_data2["scheduler"])

    start_epoch2 = checkpoint_data2["epoch"] + 1
    best_macro_acc2 = checkpoint_data2["best_macro_acc"]
    best_val_loss2 = checkpoint_data2["best_val_loss"]
    early_stop_counter2 = checkpoint_data2["early_stop_counter"]

    print(f"Resuming from epoch {start_epoch2 + 1}")

def evaluate_train2(loader):
    model2.eval()

    loss_sum2 = 0.0
    correct2 = 0
    total2 = 0
    class_correct2 = np.zeros(len(CLASS_NAMES2), dtype=np.int64)
    class_total2 = np.zeros(len(CLASS_NAMES2), dtype=np.int64)

    with torch.no_grad():
        for images, labels, _ in loader:
            images = images.to(device2)
            labels = labels.to(device2)

            outputs = model2(images)
            loss = criterion2(outputs, labels)
            predictions = outputs.argmax(dim=1)

            loss_sum2 += loss.item() * labels.size(0)
            correct2 += (predictions == labels).sum().item()
            total2 += labels.size(0)

            for class_idx2 in range(len(CLASS_NAMES2)):
                mask2 = labels == class_idx2
                class_total2[class_idx2] += mask2.sum().item()
                class_correct2[class_idx2] += (
                    ((predictions == labels) & mask2).sum().item()
                )

    class_acc2 = np.divide(
        class_correct2,
        class_total2,
        out=np.zeros(len(CLASS_NAMES2), dtype=float),
        where=class_total2 > 0
    )

    present_classes2 = class_total2 > 0
    macro_acc2 = class_acc2[present_classes2].mean()

    return (
        loss_sum2 / max(total2, 1),
        correct2 / max(total2, 1),
        macro_acc2,
        class_acc2
    )

print("Device:", device2)
print("Train samples:", len(train_dataset2))
print("Validation samples:", len(val_dataset2))
print("Test samples:", len(test_dataset2))
print("Maximum epochs:", EPOCHS2)

for epoch2 in range(start_epoch2, EPOCHS2):
    model2.train()

    loss_sum2 = 0.0
    correct2 = 0
    total2 = 0

    for batch_idx2, (images, labels, _) in enumerate(train_loader2):
        images = images.to(device2)
        labels = labels.to(device2)

        optimizer2.zero_grad(set_to_none=True)

        outputs = model2(images)
        loss = criterion2(outputs, labels)

        loss.backward()
        optimizer2.step()

        loss_sum2 += loss.item() * labels.size(0)
        correct2 += (outputs.argmax(1) == labels).sum().item()
        total2 += labels.size(0)

        if (batch_idx2 + 1) % 100 == 0:
            print(
                f"Epoch {epoch2 + 1}/{EPOCHS2} | "
                f"Batch {batch_idx2 + 1}/{len(train_loader2)}"
            )

    train_loss2 = loss_sum2 / max(total2, 1)
    train_acc2 = correct2 / max(total2, 1)

    val_loss2, val_acc2, val_macro2, _ = evaluate_train2(val_loader2)
    scheduler2.step(val_loss2)

    print(
        f"\nEpoch {epoch2 + 1}/{EPOCHS2} | "
        f"Train Loss: {train_loss2:.4f} | "
        f"Train Acc: {train_acc2:.2%} | "
        f"Val Loss: {val_loss2:.4f} | "
        f"Val Acc: {val_acc2:.2%} | "
        f"Val Macro Acc: {val_macro2:.2%}"
    )

    improved2 = (
        val_macro2 > best_macro_acc2
        or (
            np.isclose(val_macro2, best_macro_acc2)
            and val_loss2 < best_val_loss2
        )
    )

    if improved2:
        best_macro_acc2 = val_macro2
        best_val_loss2 = val_loss2
        early_stop_counter2 = 0

        torch.save(model2.state_dict(), MODEL_BEST2)
        print("Best model saved.")
    else:
        early_stop_counter2 += 1

    torch.save({
        "epoch": epoch2,
        "model": model2.state_dict(),
        "optimizer": optimizer2.state_dict(),
        "scheduler": scheduler2.state_dict(),
        "best_macro_acc": best_macro_acc2,
        "best_val_loss": best_val_loss2,
        "early_stop_counter": early_stop_counter2,
    }, CHECKPOINT2)

    if early_stop_counter2 >= PATIENCE2:
        print("Early stopping.")
        break

if not MODEL_BEST2.exists():
    raise FileNotFoundError("Chưa có best model checkpoint.")

model2.load_state_dict(
    torch.load(MODEL_BEST2, map_location=device2, weights_only=True)
)

test_loss2, test_acc2, test_macro2, test_class_acc2 = evaluate_train2(
    test_loader2
)

print("\nFINAL TEST RESULTS")
print(f"Test Loss: {test_loss2:.4f}")
print(f"Test Accuracy: {test_acc2:.2%}")
print(f"Test Macro Accuracy: {test_macro2:.2%}")

print("\nAccuracy by class:")
for i, class_name in enumerate(CLASS_NAMES2):
    print(f"{class_name}: {test_class_acc2[i]:.2%}")

print("\nBest model:", MODEL_BEST2.resolve())
print("Training checkpoint:", CHECKPOINT2.resolve())

Device: cpu
Train samples: 15912
Validation samples: 2156
Test samples: 2946
Maximum epochs: 20
Epoch 1/20 | Batch 100/995
Epoch 1/20 | Batch 200/995
Epoch 1/20 | Batch 300/995
Epoch 1/20 | Batch 400/995
Epoch 1/20 | Batch 500/995
Epoch 1/20 | Batch 600/995
Epoch 1/20 | Batch 700/995
Epoch 1/20 | Batch 800/995
Epoch 1/20 | Batch 900/995

Epoch 1/20 | Train Loss: 0.9150 | Train Acc: 69.84% | Val Loss: 0.5461 | Val Acc: 82.33% | Val Macro Acc: 83.29%
Best model saved.
Epoch 2/20 | Batch 100/995
Epoch 2/20 | Batch 200/995
Epoch 2/20 | Batch 300/995
Epoch 2/20 | Batch 400/995
Epoch 2/20 | Batch 500/995
Epoch 2/20 | Batch 600/995
Epoch 2/20 | Batch 700/995
Epoch 2/20 | Batch 800/995
Epoch 2/20 | Batch 900/995

Epoch 2/20 | Train Loss: 0.5315 | Train Acc: 81.90% | Val Loss: 0.4905 | Val Acc: 83.95% | Val Macro Acc: 84.98%
Best model saved.
Epoch 3/20 | Batch 100/995
Epoch 3/20 | Batch 200/995
Epoch 3/20 | Batch 300/995
Epoch 3/20 | Batch 400/995
Epoch 3/20 | Batch 500/995
Epoch 3/20 | Batch 

In [18]:
PLACEHOLDER = Path(r"E:\SPCK-CSI18\data\test_IMG")
MODEL_PATH2 = Path(r"garbage_classifier_4datasets_best.pth")

CLASS_NAMES2 = [
    "battery", "biological", "cardboard", "clothes", "glass",
    "metal", "paper", "plastic", "shoes", "trash"
]

IMAGE_EXTENSIONS2 = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

device2 = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

if not PLACEHOLDER.is_dir():
    raise FileNotFoundError(f"Folder not found: {PLACEHOLDER}")

if not MODEL_PATH2.is_file():
    raise FileNotFoundError(f"Model not found: {MODEL_PATH2}")

model_test2 = models.mobilenet_v3_small(weights=None)

model_test2.classifier[-1] = nn.Linear(
    model_test2.classifier[-1].in_features,
    len(CLASS_NAMES2)
)

model_test2.load_state_dict(
    torch.load(MODEL_PATH2, map_location=device2)
)

model_test2 = model_test2.to(device2)
model_test2.eval()

test_transform2 = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

image_paths2 = sorted(
    path for path in PLACEHOLDER.rglob("*")
    if path.is_file()
    and path.suffix.lower() in IMAGE_EXTENSIONS2
)

if not image_paths2:
    raise ValueError("No images found in the selected folder.")

print("Images found:", len(image_paths2))
print("Device:", device2)

with torch.no_grad():
    for image_path2 in image_paths2:
        try:
            with Image.open(image_path2) as source2:
                if (
                    "A" in source2.getbands()
                    or "transparency" in source2.info
                ):
                    rgba2 = source2.convert("RGBA")
                    background2 = Image.new(
                        "RGBA", rgba2.size, (245, 245, 245, 255)
                    )
                    image2 = Image.alpha_composite(
                        background2, rgba2
                    ).convert("RGB")
                else:
                    image2 = source2.convert("RGB")

            input2 = test_transform2(image2).unsqueeze(0).to(device2)

            output2 = model_test2(input2)
            probabilities2 = torch.softmax(output2, dim=1)[0]

            top_indices2 = torch.argsort(
                probabilities2, descending=True
            )[:3]

            predicted_idx2 = top_indices2[0].item()
            predicted_class2 = CLASS_NAMES2[predicted_idx2]
            confidence2 = probabilities2[predicted_idx2].item() * 100

            print(f"\nImage: {image_path2.name}")
            print(f"Prediction: {predicted_class2}")
            print(f"Confidence: {confidence2:.2f}%")
            print("Top 3 predictions:")

            for idx2 in top_indices2.tolist():
                print(
                    f"  {CLASS_NAMES2[idx2]}: "
                    f"{probabilities2[idx2].item() * 100:.2f}%"
                )

        except Exception as error2:
            print(f"\nFailed: {image_path2.name} | {error2}")

print("\nTesting completed.")

Images found: 29
Device: cpu

Image: 22ea6d0a39b519dc8ae39e53d5317fd2.jpg
Prediction: glass
Confidence: 100.00%
Top 3 predictions:
  glass: 100.00%
  metal: 0.00%
  plastic: 0.00%

Image: 29babe9fe7e07857b21119f39d502620.jpg
Prediction: shoes
Confidence: 100.00%
Top 3 predictions:
  shoes: 100.00%
  clothes: 0.00%
  glass: 0.00%

Image: 33f1ba59db0047774f9b5334cf059684.jpg
Prediction: cardboard
Confidence: 55.70%
Top 3 predictions:
  cardboard: 55.70%
  paper: 39.07%
  plastic: 5.17%

Image: 4632d07808c8a9fc90bd20836cd94a2d.jpg
Prediction: plastic
Confidence: 37.52%
Top 3 predictions:
  plastic: 37.52%
  cardboard: 31.48%
  paper: 21.45%

Image: 496e524ca86db4fbbe35c1a50fab55c0.jpg
Prediction: cardboard
Confidence: 86.82%
Top 3 predictions:
  cardboard: 86.82%
  paper: 7.36%
  plastic: 5.63%

Image: 4d52fe2b76f7368886362b8418b5aec4.jpg
Prediction: plastic
Confidence: 97.14%
Top 3 predictions:
  plastic: 97.14%
  cardboard: 2.46%
  trash: 0.22%

Image: 4e9699c7d9557d19532201554427e91f.j